In [16]:
import os
import re
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import KFold

# ── 1. ЗАГРУЗКА ───────────────────────────────────────────────────────────────
train_path = '/kaggle/input/competitions/vacancies-2026/train.csv'
if not os.path.exists(train_path):
  train_path = '/kaggle/input/vacancies-2026/train.csv'

test_path = '/kaggle/input/competitions/vacancies-2026/test_x.csv'
if not os.path.exists(test_path):
  test_path = '/kaggle/input/vacancies-2026/test_x.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

test_ids = test['id']

y_raw = train['salary_mean_net'].values
y = np.log1p(y_raw)

n_train = len(train)
n_test = len(test)

# ── 2. NLP: TF-IDF + RIDGE ПО ОЧИЩЕННОМУ ТЕКСТУ ──────────────────────────────
print('1/4 Обработка текстов (TF-IDF + Ridge)...')


def build_text(df):
  # Усиливаем вес названия и навыков + берем очищенное описание
  name = (
      df['name_clean']
      if 'name_clean' in df.columns
      else df.get('name', pd.Series('', index=df.index))
  ).fillna('')
  skills = df.get('key_skills_name', pd.Series('', index=df.index)).fillna('')
  desc = df.get(
      'lemmaized_wo_stopwords_raw_description', pd.Series('', index=df.index)
  ).fillna('')
  return (name + ' ') * 2 + (skills + ' ') * 2 + desc.str.slice(0, 500)


text_train = build_text(train)
text_test = build_text(test)
all_text = pd.concat([text_train, text_test], axis=0)

tfidf = TfidfVectorizer(max_features=35000, ngram_range=(1, 2), min_df=2)
X_tfidf_all = tfidf.fit_transform(all_text)
X_tfidf_tr = X_tfidf_all[:n_train]
X_tfidf_te = X_tfidf_all[n_train:]

# 5-fold Out-Of-Fold предсказания Ridge по тексту
kf = KFold(n_splits=5, shuffle=True, random_state=42)
ridge_oof = np.zeros(n_train)
ridge_test_preds = np.zeros(n_test)

for tr_i, val_i in kf.split(X_tfidf_tr, y):
  r_mod = Ridge(alpha=4.0)
  r_mod.fit(X_tfidf_tr[tr_i], y[tr_i])
  ridge_oof[val_i] = r_mod.predict(X_tfidf_tr[val_i])
  ridge_test_preds += r_mod.predict(X_tfidf_te) / 5

ridge_mape = mean_absolute_percentage_error(y_raw, np.expm1(ridge_oof))
print(f'   Только Ridge по тексту дал MAPE: {ridge_mape:.4f}')

# ── 3. ФИЧИ ДЛЯ CATBOOST ────────────────────────
print('2/4 Генерация табличных признаков...')


def extract_features(df, ridge_pred):
  df = df.copy()
  # Добавляем предсказанную текстовую зарплату как главный ориентир
  df['ridge_salary_log'] = ridge_pred

  # Длины и слова
  for col in ['name', 'key_skills_name']:
    if col in df.columns:
      s = df[col].fillna('')
      df[f'{col}_len'] = s.str.len()
      df[f'{col}_words'] = s.str.split().str.len()

  if 'key_skills_name' in df.columns:
    df['skill_count'] = df['key_skills_name'].fillna('').str.count(',') + 1
    df['skill_count'] = df['skill_count'].where(
        df['key_skills_name'].notna(), 0
    )

  # Числовой опыт
  exp_map = {
      'noExperience': 0,
      'between1And3': 2,
      'between3And6': 4.5,
      'moreThan6': 7,
  }
  if 'experience_id' in df.columns:
    df['experience_num'] = df['experience_id'].map(exp_map).fillna(-1)

  # Грейды должности
  t = (
      df['name_clean'].fillna('')
      if 'name_clean' in df.columns
      else df['name'].fillna('')
  ).str.lower()
  df['is_senior'] = t.str.contains(
      r'senior|lead|главный|старший|руководитель|директор|head|лид|ведущий|шеф',
      regex=True,
  ).astype(int)
  df['is_middle'] = t.str.contains(r'middle|мидл', regex=True).astype(int)
  df['is_junior'] = t.str.contains(
      r'junior|джун|начинающий|стажер|интерн|intern|помощник|ассистент',
      regex=True,
  ).astype(int)
  df['is_it'] = t.str.contains(
      r'developer|разработчик|программист|devops|data|аналитик|analyst|qa|python|java|frontend|backend|engineer|инженер',
      regex=True,
  ).astype(int)
  df['is_low_pay'] = t.str.contains(
      r'курьер|водитель|грузчик|уборщиц|комплектовщик|охранник|кассир',
      regex=True,
  ).astype(int)

  # Удаляем тяжелые сырые тексты
  drop_cols = [
      'name',
      'name_clean',
      'key_skills_name',
      'raw_description',
      'raw_branded_description',
      'lemmaized_wo_stopwords_raw_description',
      'lemmaized_wo_stopwords_raw_branded_description',
      'salary_mean_net',
      'id',
  ]
  return df.drop(columns=[c for c in drop_cols if c in df.columns])


X_tab = extract_features(train, ridge_oof)
test_tab = extract_features(test, ridge_test_preds)

for col in X_tab.columns:
  if X_tab[col].dtype == 'object' or str(X_tab[col].dtype) == 'bool':
    X_tab[col] = X_tab[col].fillna('missing').astype(str)
    test_tab[col] = test_tab[col].fillna('missing').astype(str)
  else:
    X_tab[col] = X_tab[col].fillna(-999)
    test_tab[col] = test_tab[col].fillna(-999)

cat_features = [c for c in X_tab.columns if X_tab[c].dtype == 'object']

# ── 4. АНСАМБЛЬ CATBOOST НА 5 ФОЛДАХ ─────────────────────────────────────────
print('3/4 Обучение ансамбля CatBoost...')
test_pool = Pool(test_tab, cat_features=cat_features)
cb_oof = np.zeros(n_train)
cb_test_preds = np.zeros(n_test)

cb_params = dict(
    iterations=1400,
    learning_rate=0.07,
    depth=6,
    loss_function='MAE',
    eval_metric='MAPE',
    random_seed=42,
    verbose=0,
    early_stopping_rounds=60,
    l2_leaf_reg=4,
    thread_count=-1,
)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_tab, y)):
  X_tr, y_tr = X_tab.iloc[train_idx], y[train_idx]
  X_va, y_val = X_tab.iloc[val_idx], y[val_idx]

  tr_p = Pool(X_tr, y_tr, cat_features=cat_features)
  va_p = Pool(X_va, y_val, cat_features=cat_features)

  model = CatBoostRegressor(**cb_params)
  model.fit(tr_p, eval_set=va_p, use_best_model=True)

  val_p = model.predict(va_p)
  cb_oof[val_idx] = val_p
  cb_test_preds += model.predict(test_pool) / 5

  print(
      f'   Фолд {fold + 1} MAPE:'
      f' {mean_absolute_percentage_error(np.expm1(y_val), np.expm1(val_p)):.4f}'
  )

# ── 5. ФИНАЛЬНЫЙ БЛЕНДИНГ ─────────────────────────────────────────
print('4/4 Финальный блендинг (CatBoost + Ridge)...')
# Бленд: 85% CatBoost + 15% чистый Ridge (сглаживает ошибки)
blend_oof = 0.85 * cb_oof + 0.15 * ridge_oof
final_oof_mape = mean_absolute_percentage_error(y_raw, np.expm1(blend_oof))
print(f'\n ИТОГОВЫЙ MAPE АНСАМБЛЯ НА ВАЛИДАЦИИ: {final_oof_mape:.4f}')

final_test_log = 0.85 * cb_test_preds + 0.15 * ridge_test_preds
final_test_salary = np.clip(np.expm1(final_test_log), 0, None)
final_test_salary = np.nan_to_num(final_test_salary, nan=np.median(y_raw))

# Сохранение сабмита
submission = pd.DataFrame({'id': test_ids, 'salary_mean_net': final_test_salary})

submission.to_csv('submission.csv', index=False)
print('Файл submission.csv успешно создан!!')

1/4 Обработка текстов (TF-IDF + Ridge)...
   Только Ridge по тексту дал MAPE: 0.2911
2/4 Генерация табличных признаков...
3/4 Обучение ансамбля CatBoost...
   Фолд 1 MAPE: 0.2292
   Фолд 2 MAPE: 0.2280
   Фолд 3 MAPE: 0.2326
   Фолд 4 MAPE: 0.2309
   Фолд 5 MAPE: 0.2295
4/4 Финальный блендинг (CatBoost + Ridge)...

🔥 ИТОГОВЫЙ MAPE АНСАМБЛЯ НА ВАЛИДАЦИИ: 0.2325
✅ submission.csv готов к отправке!
